<a href="https://colab.research.google.com/github/toche7/BasicAIAgentColab/blob/main/MCP_Simple_Colab_Lab_ConnectwithAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP —  Connect with AI Agent

**Goal:** Create one MCP tool and call it from a client. **Time:** 20–30 minutes.

Run the three code cells in order. No API key is needed.


## ขั้นตอนที่ 1: ติดตั้งไลบรารี MCP SDK
ในเซลล์ถัดไป เราจะทำการติดตั้งไลบรารี `mcp` (Model Context Protocol) เวอร์ชันตั้งแต่ `1` แต่ไม่เกิน `2` เพื่อให้เราสามารถพัฒนา Server และ Client ที่ทำงานร่วมกันตามมาตรฐาน MCP ได้

In [1]:
%pip -q install "mcp>=1,<2"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 3.1 MB/s eta 0:00:00


## ขั้นตอนที่ 2: ตรวจสอบเวอร์ชันของ MCP
ใช้คำสั่ง shell `!mcp version` เพื่อตรวจสอบว่าตัวติดตั้งระบบทำงานถูกต้องและเป็นเวอร์ชันปัจจุบัน

In [7]:
!mcp version

MCP version 1.30.0


## ขั้นตอนที่ 3: สร้าง Weather Server (MCP Server)
ในเซลล์นี้ เราจะเขียนไฟล์ `weather_server.py` เพื่อจำลองระบบสภาพอากาศ (Weather Server)
* ใช้ `FastMCP` สำหรับสร้าง Server
* ตกแต่งฟังก์ชัน `get_weather` ด้วย `@mcp.tool()` เพื่อบอกให้ระบบทราบว่านี่คือเครื่องมือ (Tool) ที่เรียกใช้งานได้
* ตัวฟังก์ชันจะรับชื่อเมือง (`city`) และส่งคืนอุณหภูมิและสภาพอากาศจำลองของเมืองนั้นๆ เช่น โตเกียว ลอนดอน หรือเมืองอื่นๆ

In [2]:
%%writefile weather_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Weather Server")

@mcp.tool()
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    # A simple mock response
    city_lower = city.lower().strip()
    if "london" in city_lower:
        return f"The weather in {city} is 15°C and Rainy."
    elif "tokyo" in city_lower:
        return f"The weather in {city} is 22°C and Sunny."
    else:
        return f"The weather in {city} is 20°C and Partly Cloudy."

if __name__ == "__main__":
    mcp.run()

Writing weather_server.py


## ขั้นตอนที่ 4: เชื่อมต่อและทดสอบเรียกใช้ MCP Server
เป็นการเชื่อมต่อ Client เข้ากับ Server ที่เราเพิ่งสร้างขึ้นมาผ่าน `stdio` (Standard Input/Output)
* ส่งคำสั่งเรียกเช็กรายการเครื่องมือทั้งหมด (`list_tools`)
* ทดสอบเรียกใช้งานฟังก์ชัน `get_weather` ด้วยเมือง "Tokyo" เพื่อดูผลลัพธ์ว่าทำงานได้ถูกต้อง

In [3]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(
    command=sys.executable,
    args=["weather_server.py"]
)

async def main():
    # We pass errlog=sys.__stderr__ to avoid Colab's standard error stream lacking fileno()
    async with stdio_client(server, errlog=sys.__stderr__) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("Tools:", [tool.name for tool in tools.tools])
            result = await session.call_tool(
                "get_weather", arguments={"city": "Tokyo"}
            )
            print(result.content[0].text)

await main()

Tools: ['get_weather']
The weather in Tokyo is 22°C and Sunny.


## ขั้นตอนที่ 5: การส่งข้อความแชทไปที่โมเดลภาษา (LLM API)
โค้ดส่วนนี้ทำหน้าที่เชื่อมต่อไปยัง API ของโมเดลภาษาภายนอก (ผ่าน Groq API คลาสย่อยของ LLM)
* ดึง API Key จากสภาพแวดล้อมหรือระบบความปลอดภัยของ Colab
* ฟังก์ชัน `chat` จะทำการส่งประวัติการสนทนาทั้งหมด และโครงสร้างของเครื่องมือ (Tools Schema) ไปให้โมเดลประมวลผล

In [8]:
"""Step 2 (Split): Part 2 - Chat and Runtime execution"""
import os
import sys
import httpx
from google.colab import userdata


URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL = "qwen/qwen3.8-27b"

try:
    api_key = userdata.get('GROQ_API_KEY')
except Exception:
    api_key = os.environ.get('GROQ_API_KEY')

def chat(messages: list[dict], tools: list[dict] | None = None) -> dict:
    """POST the conversation with tools configuration to Groq API."""
    if not api_key:
        raise ValueError("GROQ_API_KEY not found.")

    body = {"model": MODEL, "messages": messages, "max_tokens": 800}
    if tools:
        body["tools"] = tools

    headers = {"Authorization": f"Bearer {api_key}"}
    resp = httpx.post(URL, headers=headers, json=body, timeout=60)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]


## ขั้นตอนที่ 6: สร้าง Agent Loop สำหรับควบคุมขั้นตอนการทำงาน
หัวใจหลักของ Agent คือระบบ Loop นี้ (`run_agent`):
1. ส่งคำถามไปถาม LLM พร้อมแนบรายชื่อเครื่องมือที่มี (`SCHEMAS`)
2. หากโมเดลต้องการใช้เครื่องมือ (มี `tool_calls` กลับมา) มันจะหยุดตอบคำถามชั่วคราว
3. ระบบจะนำชื่อเครื่องมือและตัวแปรที่โมเดลต้องการ ส่งไปสั่งให้ฟังก์ชัน `execute` ทำงานบน MCP Server
4. ส่งผลลัพธ์กลับไปให้ LLM ประมวลผลอีกครั้ง
5. วนซ้ำขั้นตอนเดิมจนกว่าโมเดลจะได้ข้อมูลครบถ้วน และตอบผู้ใช้อย่างสมบูรณ์

In [9]:
"""Step 3 (new file): the agent loop. Call the model, run requested tools, repeat until it answers."""
def run_agent(messages: list[dict], max_steps: int = 8) -> str:
    for _ in range(max_steps):  # max_steps prevents infinite loops
        reply = chat(messages, SCHEMAS)
        calls = reply.get("tool_calls") or []
        messages.append({"role": "assistant", "content": reply.get("content") or "",
                         **({"tool_calls": calls} if calls else {})})
        if not calls:
            return messages[-1]["content"]
        for call in calls:
            fn = call["function"]
            print(f"[tool] {fn['name']}({fn['arguments']})")
            messages.append({"role": "tool", "tool_call_id": call["id"],
                             "content": execute(fn["name"], fn["arguments"])})
    return f"[stopped after {max_steps} steps without a final answer]"

## ขั้นตอนที่ 7: ทดสอบโมเดลในการทักทายเบื้องต้น
เป็นการทดสอบระบบ Agent และฟังก์ชัน `run_agent` แบบทั่วไปที่ไม่มีการเรียกใช้เครื่องมือใดๆ (กำหนด `SCHEMAS = None`)

In [11]:
# Define SCHEMAS as None since we don't have active tools defined yet
SCHEMAS = None

question = "hello how are you?"
print(run_agent([{"role": "user", "content": question}]))

Hello! I'm doing well, thank you for asking. How can I help you today?


## ขั้นตอนที่ 8: เชื่อมโยง Agent เข้ากับ MCP Server (การรันแบบครบวงจร)
นี่คือการทำงานแบบ End-to-End:
* เรียกดูเครื่องมือจริงบน `weather_server.py` มาแปลงเป็น Schema เพื่อส่งให้โมเดลภาษาได้รับรู้
* ฟังก์ชัน `execute` จะทำการเปิดการเชื่อมต่อแบบ Async ไปที่ MCP Server เพื่อดึงผลลัพธ์จริง
* ทดสอบส่งคำถาม "สภาพอากาศในโตเกียวและลอนดอนเป็นอย่างไรบ้าง?" ซึ่งตัวโมเดลจะทำการเรียกใช้เครื่องมือ `get_weather` โดยอัตโนมัติ

In [13]:
import sys
import json
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Patch asyncio to allow nesting inside Colab's running event loop
nest_asyncio.apply()

server = StdioServerParameters(
    command=sys.executable,
    args=["weather_server.py"]
)

# 1. Fetch the tools from our MCP server and convert them to the chat API schema format
SCHEMAS = []

async def fetch_schemas():
    global SCHEMAS
    async with stdio_client(server, errlog=sys.__stderr__) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_response = await session.list_tools()

            for tool in tools_response.tools:
                # Map MCP tool definition to OpenAI/Groq tool schema format
                SCHEMAS.append({
                    "type": "function",
                    "function": {
                        "name": tool.name,
                        "description": tool.description,
                        "parameters": tool.inputSchema
                    }
                })

# We can now run this coroutine on Colab's loop safely using run_until_complete
asyncio.get_event_loop().run_until_complete(fetch_schemas())
print("Loaded tool schemas:", json.dumps(SCHEMAS, indent=2))

# 2. Implement the tool execution helper that calls the MCP server dynamically
def execute(name: str, arguments_str: str) -> str:
    arguments = json.loads(arguments_str)

    async def _call():
        async with stdio_client(server, errlog=sys.__stderr__) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(name, arguments=arguments)
                return result.content[0].text

    return asyncio.get_event_loop().run_until_complete(_call())

# 3. Test the full agent loop with a weather question!
question = "What is the weather like in Tokyo and London?"
print("\n--- Agent starting execution ---")
response = run_agent([{"role": "user", "content": question}])
print("\nAgent Final Answer:", response)

Loaded tool schemas: [
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get the current weather for a given city.",
      "parameters": {
        "properties": {
          "city": {
            "title": "City",
            "type": "string"
          }
        },
        "required": [
          "city"
        ],
        "title": "get_weatherArguments",
        "type": "object"
      }
    }
  }
]

--- Agent starting execution ---
[tool] get_weather({"city":"Tokyo"})
[tool] get_weather({"city":"London"})

Agent Final Answer: Here's the current weather for both cities:

🗼 **Tokyo**: 22°C, Sunny  
🇬🇧 **London**: 15°C, Rainy


## ขั้นตอนที่ 9: คลาสเก็บประวัติการคุย (ChatSession)
สำหรับระบบแชทต่อเนื่อง เราต้องจัดเก็บประวัติการคุยทั้งหมดไว้ด้วยกัน คลาส `ChatSession` จะเก็บลิสต์ `messages` และคอยเรียกใช้ `run_agent` ต่อเนื่องกันไปเพื่อรักษาระดับความเข้าใจบริบทการสนทนา

In [14]:
class ChatSession:
    def __init__(self):
        # Keep track of the full conversation history
        self.messages = []

    def send_message(self, user_message: str) -> str:
        # 1. Append the new user query to the history
        self.messages.append({"role": "user", "content": user_message})

        # 2. Run the agent loop starting with the current history
        # Note: run_agent will append assistant replies and tool results to self.messages in place
        response = run_agent(self.messages)
        return response

## ขั้นตอนที่ 10: สร้างหน้าต่างแชทจำลอง (Interactive Chat Widget)
สร้างกล่องส่งข้อความโต้ตอบที่สวยงามผ่าน Jupyter Widgets
* มีช่องพิมพ์ข้อความ, ปุ่มกดส่ง, และหน้าต่างแสดงผลลัพธ์แบ่งสีระหว่าง ผู้ใช้ (สีฟ้า) และ โมเดล (สีเทา)
* รองรับการประมวลผลเมื่อเกิดข้อผิดพลาดในการเชื่อมต่อ (เช่น ปัญหา Rate limits)

In [15]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Initialize a fresh chat session
ui_session = ChatSession()

# HTML template for chat messages
def format_message(sender, message):
    bg_color = "#E3F2FD" if sender == "You" else "#FFEBEE" if sender == "System Error" else "#F5F5F5"
    align = "right" if sender == "You" else "left"
    margin = "margin-left: 20%;" if sender == "You" else "margin-right: 20%;"
    text_color = "color: #C62828;" if sender == "System Error" else ""
    return f'''
    <div style="text-align: {align}; margin: 5px 0;">
        <div style="display: inline-block; background-color: {bg_color}; padding: 8px 12px;
                    border-radius: 12px; max-width: 80%; text-align: left; {margin} {text_color}">
            <strong>{sender}:</strong> {message.replace('\n', '<br>')}
        </div>
    </div>
    '''

# UI Elements - Fixed overflow property for correct scrolling
output_area = widgets.Output(layout=widgets.Layout(height='300px', overflow='auto', border='1px solid #ccc', padding='10px'))
text_input = widgets.Text(placeholder='Type your message here...', layout=widgets.Layout(width='80%'))
send_button = widgets.Button(description='Send', button_style='primary', layout=widgets.Layout(width='18%'))

# Action on sending a message
def on_send_click(b):
    user_msg = text_input.value.strip()
    if not user_msg:
        return

    # Clear input immediately for better UX
    text_input.value = ''

    # Append user message to UI
    with output_area:
        display(HTML(format_message("You", user_msg)))

    # Get agent response safely
    try:
        agent_reply = ui_session.send_message(user_msg)
        # Append agent reply to UI
        with output_area:
            display(HTML(format_message("Agent", agent_reply)))
    except Exception as e:
        with output_area:
            error_msg = str(e)
            if "429" in error_msg:
                error_msg = "Rate limit reached (HTTP 429). Please wait a brief moment and try again."
            display(HTML(format_message("System Error", error_msg)))

send_button.on_click(on_send_click)
text_input.on_submit(on_send_click) # Send when pressing Enter

# Display the Layout
print("Interactive Chat Widget:")
display(widgets.VBox([output_area, widgets.HBox([text_input, send_button])]))

Interactive Chat Widget:


[tool] get_weather({"city":"Bangkok"})
[tool] get_weather({"city":"Tokyo"})
[tool] get_weather({"city":"Krabi"})
